In [1]:

import glob

import os
import pickle

import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
data_path = "/home/rajib/mTSBench/Datasets/mTSBench/SMD"
csv_files = glob.glob(os.path.join(data_path, "*test.csv"))

In [3]:
csv_files[0]

'/home/rajib/mTSBench/Datasets/mTSBench/SMD/SMD_machine-2-2_test.csv'

In [4]:
NORMAL_SIGNAL_LENGTH = 256
def load_csv_as_multivariate(csv_path: str) -> tuple[np.ndarray | None, np.ndarray | None]:
    """
    Load one *test.csv file.

    Returns
    -------
    features : float32 array (n_variates, time_steps) — timestamp/is_anomaly excluded.
    labels   : int32 array (time_steps,), 1=anomaly 0=normal (all-zero if column absent).
    """
    df = pd.read_csv(csv_path)
    print(f'df shape is -->{df.shape}')
    feature_cols = [c for c in df.columns if c not in ("timestamp", "is_anomaly")]
    if not feature_cols:
        return None, None
    try:
        features = df[feature_cols].values.T.astype(np.float32)
        labels = df["is_anomaly"].values.astype(np.int32) if "is_anomaly" in df.columns \
            else np.zeros(df.shape[0], dtype=np.int32)
        return features, labels
    except Exception as e:
        print(f"Error processing {csv_path}: {e}")
        return None, None
    
def create_pairs(
    data: np.ndarray,
    labels: np.ndarray,
    context_length: int,
    prediction_length: int,
    stride: int,
) -> list[dict]:
    """
    Slide a window over the series. For each start t (from context_length onward):

      context        = data[:, t - context_length : t]      (always full, real steps)
      future         = data[:, t : t + prediction_length]   (full window only)
      future_labels  = labels[t : t + prediction_length]    (one label per future step)

    Windows with fewer than `prediction_length` future steps remaining are skipped.
    """
    pairs = []
    total = data.shape[1]
    for t in range(context_length, total, stride):
        fut_end = t + prediction_length
        if fut_end > total:
            break                                          # not enough future steps left

        ctx = data[:, t - context_length:t].astype(np.float32, copy=False)
        fut = data[:, t:fut_end].astype(np.float32, copy=False)
        fut_labels = labels[t:fut_end].astype(np.int32, copy=False)

        pairs.append({
            "context": {"target": ctx},
            "future":  {"target": fut},
            "future_labels": fut_labels,
        })
    return pairs

def get_normal_zones(boundaries: list[tuple[int, int]], total: int) -> list[tuple[int, int]]:
    """Normal (non-anomaly) zones as (start, end) pairs."""
    zones, prev = [], 0
    for s, e in boundaries:
        if s > prev:
            zones.append((prev, s))
        prev = e
    if prev < total:
        zones.append((prev, total))
    return zones


def extract_normal_signal(
    data: np.ndarray,
    normal_zones: list[tuple[int, int]],
    length: int,
) -> np.ndarray | None:
    """
    Return a (F, length) reference normal signal sampled from the series' normal zones.

      1. If a single normal zone is long enough, take its last `length` timesteps.
      2. Otherwise concatenate normal zones (longest first) until enough.
      3. If still short, left-pad with NaN.

    Returns None if there are no normal zones at all.
    """
    if not normal_zones:
        return None

    sorted_zones = sorted(normal_zones, key=lambda z: z[1] - z[0], reverse=True)
    s, e = sorted_zones[0]
    if e - s >= length:
        return data[:, e - length:e].astype(np.float32, copy=False)

    chunks, collected = [], 0
    for s, e in sorted_zones:
        chunks.append(data[:, s:e])
        collected += e - s
        if collected >= length:
            break

    combined = np.concatenate(chunks, axis=1).astype(np.float32, copy=False)
    if combined.shape[1] >= length:
        return combined[:, -length:]

    F = combined.shape[0]
    pad = np.full((F, length - combined.shape[1]), np.nan, dtype=np.float32)
    return np.concatenate([pad, combined], axis=1)

def _attach_normal_signal(pairs: list[dict], normal_sig: np.ndarray | None) -> None:
    """In-place: attach the same per-series normal_signal reference to every pair."""
    for p in pairs:
        p["normal_signal"] = normal_sig


def pairs_to_model_inputs(pairs: list[dict]) -> list[dict]:
    """
    Convert pairs to fixed-length model inputs:

        [normal_signal (256) | context (C) | future (P)]

    Each output dict carries `future_labels` (P,) int array (0=normal, 1=anomaly),
    one label per future timestep.
    """
    out = []
    for p in pairs:
        ctx, fut = p["context"]["target"], p["future"]["target"]
        normal = p.get("normal_signal")
        if normal is None:
            normal = np.full((ctx.shape[0], NORMAL_SIGNAL_LENGTH), np.nan, dtype=np.float32)
        target = np.concatenate([normal, ctx, fut], axis=1)
        out.append({"target": target, "future_labels": p["future_labels"]})
    return out
def extract_anomaly_boundaries(labels: np.ndarray) -> list[tuple[int, int]]:
    """Contiguous anomaly regions as (start, end) with end EXCLUSIVE."""
    boundaries, in_anom, start = [], False, 0
    for i, v in enumerate(labels):
        if v == 1 and not in_anom:
            in_anom, start = True, i
        elif v == 0 and in_anom:
            in_anom = False
            boundaries.append((start, i))
    if in_anom:
        boundaries.append((start, len(labels)))
    return boundaries



In [7]:
feat, lbl = load_csv_as_multivariate(csv_files[1])

df shape is -->(11123, 40)


In [9]:
lbl.shape

(11123,)

In [11]:
min_length = 50
context_length = 512
prediction_length = 64
min_req = max(min_length, context_length + prediction_length)
all_inputs, all_labels, skipped = [], [], 0
for f in csv_files[:7]:    
    feat, lbl = load_csv_as_multivariate(f)
    all_inputs.append({"target": feat})
    all_labels.append(lbl)



df shape is -->(5459, 40)
df shape is -->(11123, 40)
df shape is -->(4930, 40)
df shape is -->(10084, 40)
df shape is -->(17131, 40)
df shape is -->(17468, 40)
df shape is -->(18188, 40)


In [19]:
all_inputs[1]['target'].shape

(38, 11123)

In [20]:
seed = 42
val_fraction =0.1
rng = np.random.default_rng(seed)
idx     = rng.permutation(len(all_inputs))
n_val   = max(1, int(len(all_inputs) * val_fraction))
val_set = set(idx[:n_val].tolist())
val_set = set(idx[:n_val].tolist())
train_inputs = [all_inputs[i] for i in range(len(all_inputs)) if i not in val_set]
val_inputs   = [all_inputs[i] for i in val_set]
train_labels = [all_labels[i] for i in range(len(all_inputs)) if i not in val_set]
val_labels   = [all_labels[i] for i in val_set]


In [33]:
stride = 576
tag = 'train'
NORMAL_SIGNAL_LENGTH = 256
out = []
for series, lbl in tqdm(zip(train_inputs, train_labels),
                                total=len(train_inputs), desc=f"Building {tag} pairs",
                                unit="series"):
    print(f"Series shape: {series['target'].shape}, Label shape: {lbl.shape}")
    pairs = create_pairs(series["target"], lbl, context_length, prediction_length, stride)
    zones = get_normal_zones(extract_anomaly_boundaries(lbl), len(lbl))
    normal_sig = extract_normal_signal(series["target"], zones, NORMAL_SIGNAL_LENGTH)
    _attach_normal_signal(pairs, normal_sig)
    out.extend(pairs)




Building train pairs: 100%|██████████| 6/6 [00:00<00:00, 141.84series/s]

Series shape: (38, 5459), Label shape: (5459,)
Series shape: (38, 11123), Label shape: (11123,)
Series shape: (38, 4930), Label shape: (4930,)
Series shape: (38, 17131), Label shape: (17131,)
Series shape: (38, 17468), Label shape: (17468,)
Series shape: (38, 18188), Label shape: (18188,)


In [17]:
out[0].keys()

dict_keys(['context', 'future', 'future_labels', 'normal_signal'])

In [18]:
out[0]['normal_signal'].shape

(38, 256)

In [35]:
def pairs_to_model_inputs(pairs: list[dict]) -> list[dict]:
    """
    Convert pairs to fixed-length model inputs:

        [normal_signal (256) | context (C) | future (P)]

    Each output dict carries `future_labels` (P,) int array (0=normal, 1=anomaly),
    one label per future timestep.
    """
    out = []
    for p in pairs:
        ctx, fut = p["context"]["target"], p["future"]["target"]
        normal = p.get("normal_signal")
        if normal is None:
            normal = np.full((ctx.shape[0], NORMAL_SIGNAL_LENGTH), np.nan, dtype=np.float32)
        target = np.concatenate([normal, ctx, fut], axis=1)
        out.append({"target": target, "future_labels": p["future_labels"]})
    return out

In [36]:
train_model_inputs = pairs_to_model_inputs(out)

In [38]:
train_model_inputs[0].keys()

dict_keys(['target', 'future_labels'])

In [40]:
train_model_inputs[0]['future_labels'].shape

(64,)

In [24]:
path  = "/home/rajib/chronos-forecasting/rajib_work_space/prepared_data_labeled/train_model_inputs.pkl"
with open(path, "rb") as f:
        data = pickle.load(f)
   

In [25]:
data[0]['future_labels']

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int32)

In [26]:
for i in range(len(data)):
    print(data[i]['future_labels'].shape)

(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,)
(64,

In [27]:
anomaly = 5
count = 0
for i in range(len(data)):
    v = data[i]['future_labels']
    ones  = int((v == 1).sum())
    zeros = int((v == 0).sum())
    # print(f"Pair {i}: shape {v.shape}, zeros={zeros}, ones={ones}")
    if ones > anomaly:
        print(f"Pair {i}: shape {v.shape}, zeros={zeros}, ones={ones}")
        count += 1
print(count)

Pair 3: shape (64,), zeros=0, ones=64
Pair 4: shape (64,), zeros=0, ones=64
Pair 5: shape (64,), zeros=49, ones=15
Pair 11: shape (64,), zeros=0, ones=64
Pair 17: shape (64,), zeros=9, ones=55
Pair 19: shape (64,), zeros=23, ones=41
Pair 35: shape (64,), zeros=0, ones=64
Pair 39: shape (64,), zeros=0, ones=64
Pair 46: shape (64,), zeros=0, ones=64
Pair 56: shape (64,), zeros=11, ones=53
Pair 72: shape (64,), zeros=27, ones=37
Pair 73: shape (64,), zeros=52, ones=12
Pair 80: shape (64,), zeros=54, ones=10
Pair 83: shape (64,), zeros=18, ones=46
Pair 88: shape (64,), zeros=29, ones=35
Pair 92: shape (64,), zeros=26, ones=38
Pair 95: shape (64,), zeros=0, ones=64
Pair 96: shape (64,), zeros=0, ones=64
Pair 100: shape (64,), zeros=0, ones=64
Pair 104: shape (64,), zeros=0, ones=64
Pair 108: shape (64,), zeros=37, ones=27
Pair 110: shape (64,), zeros=22, ones=42
Pair 123: shape (64,), zeros=0, ones=64
Pair 128: shape (64,), zeros=0, ones=64
Pair 139: shape (64,), zeros=0, ones=64
Pair 144: 

In [28]:
label = "train"
threshold =5
n_anom = 0
for d in data:
    labels = d.get("future_labels")
    if labels is not None:
        d["future_type"] = int(int(np.sum(labels)) >= threshold)
    n_anom += int(d.get("future_type", 0))
print(
    f"  {label}: threshold={threshold} ones/window -> "
    f"anomaly={n_anom}, normal={len(data) - n_anom}"
)

  train: threshold=5 ones/window -> anomaly=4486, normal=21880


In [29]:
data[0].keys()

dict_keys(['target', 'future_labels', 'future_type'])

In [30]:
normal = 0
anomaly = 0
for i in range(len(data)):
    if data[i]['future_type'] == 0:
        normal += 1
    else:
        anomaly += 1
print(f"Normal: {normal}, Anomaly: {anomaly}")

Normal: 21880, Anomaly: 4486


In [31]:
inputs =data
future_types = [int(d.get("future_type", 0)) for d in inputs]

In [32]:
len(future_types)

26366

In [33]:
import torch
def _select(inputs: dict, mask: torch.Tensor) -> dict:
        """Return a subset of the batch using a boolean mask."""
        return {
            k: v[mask] if isinstance(v, torch.Tensor) and v.shape[0] == mask.shape[0] else v
            for k, v in inputs.items()
        }